# Trump Cleaned and saved

In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import wordcloud
from textblob import TextBlob
# from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.sentiment.util import *

nltk.download('stopwords')
nltk.download('wordnet')
lm = WordNetLemmatizer()

    

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ghazalehzehtab/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ghazalehzehtab/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Start new Section with cleaned data

In [82]:
biden_tweets = pd.read_csv("biden_US_cleaned.csv")
trump_tweets = pd.read_csv("trump_US_cleaned.csv")

print(biden_tweets.shape)
biden_tweets.head()


(179575, 4)


,user_id,likes,retweet_count,cleantext
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...


In [83]:
trump_tweets.head()

,user_id,likes,retweet_count,cleantext
0,19329393.0,206.0,12.0,without doubt fourseasons debacle best part do...
1,15754281.0,56.0,16.0,president donald trump youngest daughter tiffa...
2,26642006.0,695.0,291.0,woah read article president debt donald trump ...
3,17446621.0,149.0,15.0,latest episode snl tackled donaldtrump loss co...
4,17446621.0,83.0,10.0,nataliemorales explains latinx voter supported...


In [84]:
print(trump_tweets.shape)

(209928, 4)


In [85]:

def user_activity(dataframe):
    tweet_counts = dataframe.groupby('user_id').size()
    labels = tweet_counts.apply(lambda x: 'active' if x > 5 else 'less active')
    dataframe['user_activity'] = dataframe['user_id'].map(labels)
    return dataframe

biden_tweets_activity = user_activity(biden_tweets)
trump_tweets_activity = user_activity(trump_tweets)
biden_tweets_activity.shape

(179575, 5)

In [86]:
biden_tweets_activity[biden_tweets_activity['user_activity'] == 'active'].head()

,user_id,likes,retweet_count,cleantext,user_activity
7,17446621.0,406.0,40.0,victory speech joebiden said understood trump ...,active
8,17446621.0,272.0,18.0,gabrielleunion reacted news joebiden taken lea...,active
9,17446621.0,515.0,38.0,hollywood star cheer joebiden take lead pennsy...,active
10,17446621.0,186.0,15.0,gabrielle union kerry washington eva longoria ...,active
11,17446621.0,73.0,1.0,jimmykimmel offered view last day gone close r...,active


In [87]:
print("Biden active user",biden_tweets_activity[biden_tweets_activity['user_activity'] == 'active'].shape)
print("Biden less active user",biden_tweets_activity[biden_tweets_activity['user_activity'] == 'less active'].shape)

Biden active user (96705, 5)
Biden less active user (82870, 5)


In [88]:
print("Trump active user",trump_tweets_activity[trump_tweets_activity['user_activity'] == 'active'].shape)
print("Trump less active user",trump_tweets_activity[trump_tweets_activity['user_activity'] == 'less active'].shape)

Trump active user (144606, 5)
Trump less active user (65322, 5)


In [89]:
# import matplotlib.pyplot as plt
# 
# # Assuming your DataFrame is named biden_tweets_activity
# # Assuming 'user_activity' column contains activity levels
# 
# # Filter DataFrame for active and less active users
# active_users = biden_tweets_activity[biden_tweets_activity['user_activity'] == 'active']
# less_active_users = biden_tweets_activity[biden_tweets_activity['user_activity'] == 'less active']
# 
# # Plotting histogram
# plt.hist([active_users['user_activity'], less_active_users['user_activity']], 
#          color=['blue', 'red'], 
#          label=['Active Users', 'Less Active Users'])
# 
# # Customize labels and title
# plt.xlabel('Some Column')
# plt.ylabel('Frequency')
# plt.title('Comparison of Active and Less Active Users')
# plt.legend()
# 
# # Show plot
# plt.show()


In [90]:
def getsubjectivity(text):
    return TextBlob(text).sentiment.subjectivity
def getpolarity(text):
    return TextBlob(text).sentiment.polarity
def getAnalysis(score):
    if score <0:
        return 'negative'
    elif score == 0:
        return 'neutral'
    else:
        return 'positive'

In [91]:
def sentiment_analysis(tweets):
    tweets['subjectivity'] = tweets['cleantext'].apply(getsubjectivity)
    
    tweets['polarity'] = tweets['cleantext'].apply(getpolarity)
    
    tweets['analysis'] = tweets['polarity'].apply(getAnalysis)
    return tweets
#print("This is trump tweets analysis", trump_tweets.analysis.value_counts(normalize=True) * 100)


In [92]:
def check_and_convert(value):
    if isinstance(value, float):  # Check if the value is a float
        return str(value)         # Convert float to string
    else:
        return value              # Return value unchanged if not a float

# Apply the function to the DataFrame column
trump_tweets_activity['cleantext'] = trump_tweets_activity['cleantext'].apply(lambda x: check_and_convert(x))


In [93]:
trump_df = sentiment_analysis(trump_tweets_activity)
trump_df.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
0,19329393.0,206.0,12.0,without doubt fourseasons debacle best part do...,less active,0.483333,0.333333,positive
1,15754281.0,56.0,16.0,president donald trump youngest daughter tiffa...,less active,0.000000,0.000000,neutral
2,26642006.0,695.0,291.0,woah read article president debt donald trump ...,less active,0.000000,0.000000,neutral
3,17446621.0,149.0,15.0,latest episode snl tackled donaldtrump loss co...,active,0.800000,-0.033333,negative
4,17446621.0,83.0,10.0,nataliemorales explains latinx voter supported...,active,0.627778,-0.050000,negative


In [94]:
biden_df = sentiment_analysis(biden_tweets_activity)
biden_df.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...,less active,0.0000,0.00000,neutral
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...,less active,0.6625,0.49375,positive
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...,less active,0.7500,0.80000,positive
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...,less active,0.1000,0.00000,neutral
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...,less active,0.0000,0.00000,neutral


In [95]:
biden_df.head(10)

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...,less active,0.000000,0.000000,neutral
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...,less active,0.662500,0.493750,positive
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...,less active,0.750000,0.800000,positive
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...,less active,0.100000,0.000000,neutral
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...,less active,0.000000,0.000000,neutral
5,11912362.0,8855.0,784.0,president elect joebiden message http co mvsdl...,less active,0.900000,0.800000,positive
6,11912362.0,5909.0,806.0,presidenttrump release statement following pre...,less active,0.466667,0.533333,positive
7,17446621.0,406.0,40.0,victory speech joebiden said understood trump ...,active,0.750000,-0.750000,negative
8,17446621.0,272.0,18.0,gabrielleunion reacted news joebiden taken lea...,active,0.607143,0.417143,positive
9,17446621.0,515.0,38.0,hollywood star cheer joebiden take lead pennsy...,active,0.000000,0.000000,neutral


In [99]:
sorted_biden = biden_df.sort_values(by=['likes', 'retweet_count'], ascending=False)
sorted_biden.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
102,139823781.0,165702.0,17652.0,tonight woman audience trump town hall began q...,active,0.5125,0.483333,positive
19865,38362690.0,74528.0,20615.0,america assembled joebiden kamalaharris barack...,less active,0.0000,0.000000,neutral
94,139823781.0,59403.0,7489.0,god bette scared god fascist whitehouse racist...,active,0.4500,0.080000,positive
90,139823781.0,59345.0,2838.0,wonderful speech tonight president vice presid...,active,0.6800,0.522500,positive
92,139823781.0,45540.0,6133.0,god bette scared fascist whitehouse racist cri...,active,0.5625,0.100000,positive


In [101]:
biden_df.groupby('user_id')['polarity'].mean()

user_id
1.081000e+03    0.000000
2.654000e+03    0.083333
3.652000e+03    0.000000
4.358000e+03    0.000000
5.608000e+03    0.000000
                  ...   
1.325492e+18    0.000000
1.325502e+18    0.500000
1.325533e+18    0.250000
1.325548e+18   -0.255556
1.325550e+18    0.566667
Name: polarity, Length: 61201, dtype: float64

In [104]:
biden_df.analysis.value_counts()

analysis
neutral     83859
positive    65654
negative    30062
Name: count, dtype: int64

In [107]:
trump_df.analysis.value_counts()

analysis
neutral     90575
positive    71600
negative    47753
Name: count, dtype: int64